In [0]:
pip install jinja2

In [0]:
%restart_python

In [0]:
parameters = [{ "table" : "spotify_cata.silver.factstream",
                "alias" : "factstream",
                "cols"  : "factstream.stream_id , factstream.listen_duration"

}
,{ 
    "table" : "spotify_cata.silver.dimuser",
    "alias" : "dimuser",
    "cols"  : "dimuser.user_id , dimuser.user_name",
    "condition" : "factstream.user_id = dimuser.user_id"
   
},
{
    "table" : "spotify_cata.silver.dimtrack",
    "alias" : "dimtrack",
    "cols"  : "dimtrack.track_id , dimtrack.track_name",
    "condition" : "factstream.track_id = dimtrack.track_id"
}

]

In [0]:
query_text = """
            SELECT 
                {% for params in parameters %}
                    {{ params.cols }}
                        {% if not loop.last %}
                            ,
                        {% endif %}
                {% endfor %}
            FROM 
                {% for params in parameters %}
                    {% if loop.first %}
                        {{ params.table }} AS {{ params.alias }}
                    {% endif %}
                {% endfor %}
                {% for params in parameters %}
                    {% if not loop.first %}
                        LEFT JOIN {{ params.table }} AS {{ params.alias }}
                         ON {{ params.condition }}
                    {% endif %}
                {% endfor %}


"""

In [0]:
from jinja2 import Template

jinja_sql_qry = Template(query_text)
query = jinja_sql_qry.render(parameters=parameters)
print(query)

In [0]:
display(spark.sql(query))